In [ ]:
from google.colab import drive
import pandas as pd
from lxml import etree
import os

drive.mount('/content/drive')

# Sesuaikan path dengan folder di foto kamu
path = '/content/drive/My Drive/Colab Notebooks/datasetpan/'
corpus_file = path + 'pan12-sexual-predator-identification-test-corpus-2012-05-17.xml'
gt_file_prob1 = path + 'pan12-sexual-predator-identification-groundtruth-problem1.txt'
gt_file_prob2 = path + 'pan12-sexual-predator-identification-groundtruth-problem2.txt'


Mounted at /content/drive


In [ ]:
import csv

# 1. Load ID Predator (problem1)
predator_ids = set()
with open(gt_file_prob1, 'r') as f:
    for line in f:
        predator_ids.add(line.strip())

# 2. Load Ground Truth Grooming (problem2)
# Kita simpan dalam dictionary dengan format {conv_id: [list_of_lines]}
# Ini jauh lebih akurat daripada hanya mencocokkan teks
grooming_dict = {}
with open(gt_file_prob2, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            cid, lnum = parts[0], parts[1]
            if cid not in grooming_dict:
                grooming_dict[cid] = set()
            grooming_dict[cid].add(lnum)

# 3. Proses XML ke CSV Baru
output_csv_strict = path + 'pan12_strict_labels.csv'

with open(output_csv_strict, 'w', newline='', encoding='utf-8') as f_out:
    writer = csv.writer(f_out)
    writer.writerow(['conv_id', 'line', 'author', 'text', 'is_predator', 'label'])

    context = etree.iterparse(corpus_file, events=('end',), tag='conversation')

    for event, conv in context:
        conv_id = conv.get('id')

        # Ambil daftar baris yang dilabeli grooming untuk ID percakapan ini saja
        target_lines = grooming_dict.get(conv_id, set())

        for msg in conv.findall('message'):
            line_num = msg.get('line')
            author_id = msg.find('author').text if msg.find('author') is not None else ""
            text_content = msg.find('text').text if msg.find('text') is not None else ""

            # CEK 1: Apakah pengirim predator? (Berdasarkan problem1)
            is_pred = 1 if author_id in predator_ids else 0

            # CEK 2: Apakah baris ini terdaftar di problem2? (Sangat Ketat)
            # Label 1 HANYA jika line_num ada di daftar target_lines untuk conv_id ini
            label = 1 if line_num in target_lines else 0

            writer.writerow([conv_id, line_num, author_id, text_content, is_pred, label])

        conv.clear()
        while conv.getprevious() is not None:
            del conv.getparent()[0]

print("CSV Berhasil Dibuat dengan Label Strict!")

CSV Berhasil Dibuat dengan Label Strict!


In [ ]:
import pandas as pd

# 1. Baca file CSV yang sudah diproses secara strict
df_strict = pd.read_csv(output_csv_strict)

# 2. Tampilkan 20 baris pertama untuk melihat struktur kolomnya
print("--- STRUKTUR DAN 20 BARIS PERTAMA CSV ---")
display(df_strict.head(20))

# 3. Tampilkan informasi teknis (Tipe data per kolom)
print("\n--- INFORMASI DATASET ---")
print(df_strict.info())

# 4. Tampilkan statistik sederhana label
print("\n--- JUMLAH DATA PER LABEL ---")
print(df_strict['label'].value_counts())

--- STRUKTUR DAN 20 BARIS PERTAMA CSV ---


,conv_id,line,author,text,is_predator,label
0,affc2df0951b733d14ba92d19d9b7695,1,0a39f78bcb297ab0ebe8a29c28bfed89,bugmail: [Bug 6978] New: Mark eof-terminated s...,0,0
1,affc2df0951b733d14ba92d19d9b7695,2,60659cfda992013e610f285c46692d28,"Henri, can I ask you a Firefox build question ...",0,0
2,affc2df0951b733d14ba92d19d9b7695,3,b8810fee2f4a71f849f3f7409546d1d9,"60659cfda992013e610f285c46692d28: sure, but I ...",0,0
3,affc2df0951b733d14ba92d19d9b7695,4,60659cfda992013e610f285c46692d28,"It appears the build runs through, it creates ...",0,0
4,affc2df0951b733d14ba92d19d9b7695,5,60659cfda992013e610f285c46692d28,"when I start it, I get my standard install of ...",0,0
5,affc2df0951b733d14ba92d19d9b7695,6,60659cfda992013e610f285c46692d28,"Same if I make a package, unzip it, and start ...",0,0
6,affc2df0951b733d14ba92d19d9b7695,7,b8810fee2f4a71f849f3f7409546d1d9,60659cfda992013e610f285c46692d28: do you alrea...,0,0
7,affc2df0951b733d14ba92d19d9b7695,8,60659cfda992013e610f285c46692d28,Likely,0,0
8,affc2df0951b733d14ba92d19d9b7695,9,60659cfda992013e610f285c46692d28,So do I need to close all instances?,0,0
9,affc2df0951b733d14ba92d19d9b7695,10,60659cfda992013e610f285c46692d28,...other...,0,0



--- INFORMASI DATASET ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2058781 entries, 0 to 2058780
Data columns (total 6 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   conv_id      object
 1   line         int64 
 2   author       object
 3   text         object
 4   is_predator  int64 
 5   label        int64 
dtypes: int64(3), object(3)
memory usage: 94.2+ MB
None

--- JUMLAH DATA PER LABEL ---
label
0    2052303
1       6478
Name: count, dtype: int64


In [ ]:
import pandas as pd

# 1. Baca file CSV utama yang sudah kita buat tadi
df_strict = pd.read_csv(output_csv_strict)

# 2. Cari daftar conv_id yang memiliki setidaknya satu baris dengan label 1
grooming_ids = df_strict[df_strict['label'] == 1]['conv_id'].unique()

# 3. Filter DataFrame utama: Ambil semua baris yang conv_id-nya ada di daftar grooming_ids
# Ini akan mengambil satu percakapan utuh (dari chat normal sampai grooming-nya)
df_grooming_only_final = df_strict[df_strict['conv_id'].isin(grooming_ids)].copy()

# 4. Urutkan berdasarkan conv_id dan line agar alur chat tetap benar
df_grooming_only_final['line'] = pd.to_numeric(df_grooming_only_final['line'])
df_grooming_only_final = df_grooming_only_final.sort_values(['conv_id', 'line'])

# 5. Simpan ke file CSV baru
output_grooming_only = path + 'pan12_grooming_conversations_only.csv'
df_grooming_only_final.to_csv(output_grooming_only, index=False)

# --- Verifikasi Hasil ---
print(f"✅ Berhasil memisahkan percakapan grooming!")
print(f"Total Percakapan: {df_grooming_only_final['conv_id'].nunique()}")
print(f"Total Baris Pesan: {len(df_grooming_only_final)}")
print(f"File disimpan di: {output_grooming_only}")

# Tampilkan 10 baris pertama dari file baru ini
display(df_grooming_only_final.head(10))

✅ Berhasil memisahkan percakapan grooming!
Total Percakapan: 834
Total Baris Pesan: 75119
File disimpan di: /content/drive/My Drive/Colab Notebooks/datasetpan/pan12_grooming_conversations_only.csv


,conv_id,line,author,text,is_predator,label
1581178,001642060867dc1119343316fc21926c,1,8596c082ca02cf3d2e40682389f76a47,hi,1,0
1581179,001642060867dc1119343316fc21926c,2,4423d67017f371bb5b7218053f06def1,hi :),0,0
1581180,001642060867dc1119343316fc21926c,3,4423d67017f371bb5b7218053f06def1,how r u?,0,0
1581181,001642060867dc1119343316fc21926c,4,8596c082ca02cf3d2e40682389f76a47,wanna play on phone,1,0
1581182,001642060867dc1119343316fc21926c,5,4423d67017f371bb5b7218053f06def1,lolz i cant rite now,0,0
1581183,001642060867dc1119343316fc21926c,6,8596c082ca02cf3d2e40682389f76a47,ok,1,0
1581184,001642060867dc1119343316fc21926c,7,4423d67017f371bb5b7218053f06def1,im posed 2 b in bed :D,0,0
1581185,001642060867dc1119343316fc21926c,8,4423d67017f371bb5b7218053f06def1,did u play in ur band this weekend?,0,0
1581186,001642060867dc1119343316fc21926c,9,8596c082ca02cf3d2e40682389f76a47,ya i did and im super horny,1,0
1581187,001642060867dc1119343316fc21926c,10,4423d67017f371bb5b7218053f06def1,u open for any1?,0,0


In [ ]:
import pandas as pd
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords

# 1. Download stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# 2. Baca file grooming only
df_grooming = pd.read_csv(path + 'pan12_grooming_conversations_only.csv')

# 3. Ambil teks khusus Label 1 (Tindakan Grooming)
text_label_1 = df_grooming[df_grooming['label'] == 1]['text'].astype(str).tolist()

# 4. Fungsi pembersihan teks
def clean_text(text):
    text = text.lower()
    # Hapus simbol tapi biarkan tanda tanya karena sering dipakai dalam ajakan
    text = re.sub(r'[^a-z\s\?]', '', text)
    words = text.split()
    # Kita tetap hapus stopwords agar kata-kata yang bermakna (seperti 'sex', 'photo') terlihat
    words = [w for w in words if w not in stop_words and len(w) > 1]
    return words

# 5. Hitung frekuensi
all_words_1 = []
for t in text_label_1:
    all_words_1.extend(clean_text(t))

word_freq_1 = Counter(all_words_1)

# 6. Tampilkan 30 kata teratas
print("="*60)
print("🔥 TOP 30 KATA KUNCI PADA LABEL 1 (GROOMING ACTION)")
print("="*60)

df_freq_1 = pd.DataFrame(word_freq_1.most_common(30), columns=['Kata Kunci', 'Frekuensi'])
print(df_freq_1)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


🔥 TOP 30 KATA KUNCI PADA LABEL 1 (GROOMING ACTION)
   Kata Kunci  Frekuensi
0        want        623
1       would        583
2        like        549
3          ur        419
4         get        410
5          im        333
6         see        266
7         sex        263
8        dont        239
9       think        230
10        lol        225
11       love        216
12      could        214
13       know        212
14      pussy        210
15        cum        191
16       make        185
17       sexy        179
18       cock        175
19      wanna        169
20       kiss        167
21      naked        165
22       baby        162
23      going        151
24        one        141
25       well        139
26         go        133
27       feel        128
28       ever        127
29       suck        127


In [ ]:
# 1. Daftar kata kunci berbahaya (VERSI LENGKAP)
# Ditambah: Slang (hard, dck), Aktivitas (sleep, bed, strip),
# serta istilah manipulasi (age, video, send)
danger_keywords = [
    # Anatomi & Seksual (Asli + Tambahan)
    'pussy', 'cum', 'cock', 'naked', 'suck', 'sex', 'sexy', 'kiss', 'horny',
    'dick', 'boobs', 'breast', 'nude', 'masturbat', 'dick', 'vagina', 'penis',

    # Aktivitas & Ajakan
    'cam', 'webcam', 'private', 'strip', 'sleep', 'bed',
    'touch', 'feeling', 'wet', 'hard', 'moan',

    # Media & Permintaan
    'pic', 'photo', 'video', 'send', 'snap', 'webcam',

    # Manipulasi/Konteks Dewasa
    'baby', 'love', 'hot', 'older', 'young', 'age', 'mature'
]

def final_labeling(row):
    # Jika pakar sudah bilang 1, kita tetap ikut 1
    if row['label'] == 1:
        return 1

    # Jika pakar bilang 0, tapi ada kata berbahaya dari predator, kita ubah jadi 1
    if row['is_predator'] == 1:
        msg = str(row['text']).lower()
        # Kita gunakan set agar pencarian kata lebih cepat dan akurat
        if any(word in msg for word in danger_keywords):
            return 1

    return 0

# 2. Terapkan logika baru
df_grooming_only_final['label_final'] = df_grooming_only_final.apply(final_labeling, axis=1)

# 3. Lihat perubahannya
print("="*50)
print("📊 PERBANDINGAN LABEL: HEURISTIC SUPER LENGKAP")
print("="*50)
print(df_grooming_only_final['label_final'].value_counts())
print("-" * 50)

naik_kelas = ((df_grooming_only_final['label'] == 0) & (df_grooming_only_final['label_final'] == 1)).sum()
print(f"Jumlah label 0 yang berubah jadi 1: {naik_kelas} baris")

📊 PERBANDINGAN LABEL: HEURISTIC SUPER LENGKAP
label_final
0    65317
1     9802
Name: count, dtype: int64
--------------------------------------------------
Jumlah label 0 yang berubah jadi 1: 3324 baris


In [ ]:
# 1. Tentukan path dan nama file baru
output_file_final = path + 'datasetfinal.csv'

# 2. Simpan DataFrame ke CSV
# Kita simpan semua kolom agar kamu tetap bisa membandingkan 'label' asli dan 'label_final'
df_grooming_only_final.to_csv(output_file_final, index=False)

print(f"✅ Berhasil! Dataset dengan label heuristic telah disimpan.")
print(f"📁 Lokasi: {output_file_final}")
print(f"📊 Total Baris: {len(df_grooming_only_final)}")
print(f"🔥 Total Label 1 (Grooming): {df_grooming_only_final['label_final'].sum()}")

✅ Berhasil! Dataset dengan label heuristic telah disimpan.
📁 Lokasi: /content/drive/My Drive/Colab Notebooks/datasetpan/datasetfinal.csv
📊 Total Baris: 75119
🔥 Total Label 1 (Grooming): 9802


In [ ]:
# 1. Pilih kolom yang diinginkan saja
# Kita ambil label_final sebagai kolom 'label' yang baru
df_final_export = df_grooming_only_final[['conv_id', 'line', 'author', 'text', 'label_final']].copy()

# 2. Rename 'label_final' menjadi 'label' agar sesuai format yang kamu mau
df_final_export = df_final_export.rename(columns={'label_final': 'label'})

# 3. Simpan ke CSV
output_path = path + 'dataset_heuristic.csv'
df_final_export.to_csv(output_path, index=False, sep='\t') # Menggunakan tab agar rapi, atau ganti ',' untuk comma separated

print(f"✅ File berhasil dibuat dengan format ringkas!")
print(f"📁 Lokasi: {output_path}")
print("-" * 30)
# Menampilkan 5 baris pertama untuk memastikan formatnya pas
display(df_final_export.head())

✅ File berhasil dibuat dengan format ringkas!
📁 Lokasi: /content/drive/My Drive/Colab Notebooks/datasetpan/dataset_heuristic.csv
------------------------------


,conv_id,line,author,text,label
1581178,001642060867dc1119343316fc21926c,1,8596c082ca02cf3d2e40682389f76a47,hi,0
1581179,001642060867dc1119343316fc21926c,2,4423d67017f371bb5b7218053f06def1,hi :),0
1581180,001642060867dc1119343316fc21926c,3,4423d67017f371bb5b7218053f06def1,how r u?,0
1581181,001642060867dc1119343316fc21926c,4,8596c082ca02cf3d2e40682389f76a47,wanna play on phone,0
1581182,001642060867dc1119343316fc21926c,5,4423d67017f371bb5b7218053f06def1,lolz i cant rite now,0
